In [5]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import statistics

In [6]:
RETRIEVAL_DIR = Path("../output/retrieval")
DATA_DIR = Path("../data/gold_data")

retrieval_files = list((RETRIEVAL_DIR/"bm25_rerank").rglob("*.json"))
gold_full_df = pd.read_csv(DATA_DIR / "gold_standard_processed.csv")

# Create Dataset

In [7]:
gold_pages_by_report = gold_full_df.groupby("report_id")["page"].agg(lambda x: set(x.dropna())).to_dict()

results = []

for retrieval_file in retrieval_files:
    # if retrieval_file.stem.split("_")[0] != "apollo medical holdings":
    #     continue
    with open(retrieval_file, "r", encoding="utf-8") as f:
        rdata = json.load(f)
    
    
    retrieval_method = rdata.get("method")
    
    report_id = rdata.get("report_id")
    
    retrieved_results = rdata.get("items", [])
    retrieved_pages = set()
    retrieved_scores = []
    
    for item in retrieved_results:
        metadata = item.get("metadata", {})
        page_numbers = metadata.get("page_no")
        score = item.get("score")
        retrieved_pages.add(page_numbers)
        retrieved_scores.append(score)
        retrieved_scores.sort()
    
    gold_pages = gold_pages_by_report[report_id]
    
    if gold_pages and gold_pages.issubset(retrieved_pages):
        found_all_gold_pages = True
        gold_exists = True
    elif gold_pages:
        found_all_gold_pages = False
        gold_exists = True
    else:
        found_all_gold_pages = False
        gold_exists = False
    
    results.append({
        "report_id": report_id,
        "1st": retrieved_scores[-1] if len(retrieved_scores) > 0 else None,
        "2nd": retrieved_scores[-2] if len(retrieved_scores) > 1 else None,
        "3rd": retrieved_scores[-3] if len(retrieved_scores) > 2 else None,
        "4th": retrieved_scores[-4] if len(retrieved_scores) > 3 else None,
        "5th": retrieved_scores[-5] if len(retrieved_scores) > 4 else None,
        "mean_score": statistics.mean(retrieved_scores) if retrieved_scores else None,
        "score_gap": retrieved_scores[-1] - retrieved_scores[-2] if len(retrieved_scores) > 1 else None,
        "found_all_gold_pages": found_all_gold_pages,
        "gold_exists": gold_exists,
    })
    
df = pd.DataFrame(results)
df

,report_id,1st,2nd,3rd,4th,5th,mean_score,score_gap,found_all_gold_pages,gold_exists
0,acuity brands inc_2022,0.982014,0.904651,0.808067,0.577495,0.531209,0.760687,0.077363,True,True
1,addtech_2022,0.468791,0.307358,0.191933,0.132964,0.080357,0.236281,0.161433,True,True
2,aixtron_2020,0.966914,0.766294,0.222700,0.222700,0.125923,0.460906,0.200620,False,True
3,allfunds group_2021,0.798187,0.268941,0.173288,0.071591,0.031144,0.268630,0.529245,True,True
4,Allianz_2022,0.955319,0.718594,0.268941,0.233706,0.191933,0.473699,0.236725,True,True
...,...,...,...,...,...,...,...,...,...,...
134,varta ag_2021,0.766294,0.090093,0.042088,0.016915,0.016915,0.186461,0.676201,True,True
135,webuild_2019,0.743168,0.320821,0.085099,0.035145,0.007121,0.238271,0.422347,True,True
136,western union_2019,0.156105,0.060087,0.047426,0.017986,0.014957,0.059312,0.096018,False,False
137,xvivo perfusion_2020,0.005555,0.003377,0.003377,0.002473,0.002473,0.003451,0.002178,False,False


In [8]:
df[df["gold_exists"] == False]

,report_id,1st,2nd,3rd,4th,5th,mean_score,score_gap,found_all_gold_pages,gold_exists
5,alumina ltd_2022,0.007577,0.007121,0.006693,0.004070,0.002981,0.005689,0.000456,False,False
6,amano corp_2022,0.031144,0.021615,0.019124,0.013223,0.013223,0.019666,0.009528,False,False
9,apollo medical holdings_2022,0.002632,0.001927,0.001755,0.001648,0.000911,0.001775,0.000705,False,False
10,arbonia ag_2018,0.001810,0.001549,0.001367,0.001284,0.001207,0.001443,0.000261,False,False
12,asos plc_2017,0.007121,0.001755,0.001455,0.001325,0.001134,0.002558,0.005367,False,False
...,...,...,...,...,...,...,...,...,...,...
131,tricon residential inc_2020,0.014957,0.010987,0.009126,0.003945,0.003824,0.008568,0.003970,False,False
132,trisura group_2022,0.003707,0.001755,0.000940,0.000646,0.000607,0.001531,0.001953,False,False
136,western union_2019,0.156105,0.060087,0.047426,0.017986,0.014957,0.059312,0.096018,False,False
137,xvivo perfusion_2020,0.005555,0.003377,0.003377,0.002473,0.002473,0.003451,0.002178,False,False
